In [ ]:
import pandas as pd
import numpy as np

# Load data in chunks to avoid memory error
chunk_size = 100000
chunks = []

for chunk in pd.read_csv("../scripts/data/AirNow_AQS_Meteo_Joined_Jan2020-Oct2025.csv", chunksize=chunk_size):
    chunks.append(chunk)

df = pd.concat(chunks, ignore_index=True)

df.head(10)

## Change Column Names for easier analysis and processing

In [ ]:
df.rename(columns={'0' : 'latitude', '1': 'longitude', '2': 'utc', '3': 'parameter', '4': 'concentration',
                   '5': "unit", '6': "sitename", '7': "agencyname", '8': "fullaqscode", '9': "intlaqscode"}, inplace=True)

shape = df.shape
print(f"{shape}")

df.head(10)

### Keep only relevant columns

In [ ]:
df = df.drop(columns=[
    'Latitude',
    'Longitude',
    'utc',
    'UTC_hour',
    'sitename',
    'agencyname',
    'fullaqscode',
    'intlaqscode',
    'Datum',
    'Elevation',
    'Land Use',
    'Location Setting',
    'Site Established Date',
    'Site Closed Date',
    'Met Site State Code',
    'Met Site County Code',
    'Met Site Site Number',
    'Met Site Type',
    'Met Site Distance',
    'Met Site Direction',
    'GMT Offset',
    'Owning Agency',
    'Local Site Name',
    'Address',
    'Zip Code',
    'State Name',
    'County Name',
    'City Name',
    'CBSA Name',
    'Tribe Name',
    'Extraction Date',
    'State Code',
    'County Code',
    'Site Number',
    'unit'
])

shape = df.shape
print(f"{shape}")

df.head(10)

### Pivot the data to have pollutants as columns

In [ ]:
df = df.pivot_table(index=['latitude', 'longitude', 'UTC', 'temperature_2m', 'precipitation', 'weather_code', 'wind_speed_10m', 'wind_direction_10m', 'wind_gusts_10m', 'relative_humidity_2m'], columns='parameter', values='concentration').reset_index()

df.columns.name = None  # Remove the columns name

shape = df.shape
print(f"{shape}")

df.head(10)

## Clean negative values

In [ ]:
pollutants = ['PM2.5', 'OZONE', 'NO2', 'SO2', 'CO', 'PM10']

for pollutant in pollutants:
    if pollutant in df.columns:
        df[pollutant] = df[pollutant].mask(df[pollutant] < 0, np.nan)

df.to_pickle("../scripts/data/checkpoint_after_cleaning.pkl")
